In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
PIPELINE_NAME = "Gold Customer Segmentation"
SOURCE_TABLE = GOLD_CUSTOMER_360
TARGET_TABLE = GOLD_CUSTOMER_SEGMENTATION
RUN_ID = generate_run_id()
START_TIME = datetime.now()

In [0]:
print("GOLD CUSTOMER SEGMENTATION PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD CUSTOMER SEGMENTATION PIPELINE
Pipeline : Gold Customer Segmentation
Run ID : 7524cacf-0087-4c24-9d44-ee775afa7595
Target : retailmart.gold.customer_segmentation


In [0]:
customer_360_df = spark.table(SOURCE_TABLE)
display(customer_360_df.limit(10))

customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415
CUST_000066,Campo Grande,MS,4,4,4156.87,1039.2175,2021-06-09T23:00:00.000Z,2022-06-14T08:46:00.000Z,370
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741
CUST_000095,Teresina,PI,1,1,1036.53,1036.53,2022-08-08T15:08:00.000Z,2022-08-08T15:08:00.000Z,0
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559
CUST_000117,Manaus,AM,0,0,0.0,0.0,null,null,0
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849
CUST_000136,Joao Pessoa,PB,1,3,3487.1,1162.3666666666666,2021-06-08T11:29:00.000Z,2021-06-08T11:29:00.000Z,0
CUST_000159,Porto Velho,RO,2,2,2271.27,1135.635,2021-11-11T15:22:00.000Z,2022-06-21T01:37:00.000Z,222


In [0]:
print(f"Total Customers : {customer_360_df.count()}")
customer_360_df.printSchema()

Total Customers : 15000
root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_purchased: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- average_item_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- customer_lifetime_days: integer (nullable = true)



In [0]:
spark.sql(f"""
SELECT

    MIN(total_spent) AS min_spent,
    MAX(total_spent) AS max_spent,
    AVG(total_spent) AS avg_spent

FROM {GOLD_CUSTOMER_360}
""").show()

+---------+---------+-----------------+
|min_spent|max_spent|        avg_spent|
+---------+---------+-----------------+
|      0.0| 37171.46|7486.081435999996|
+---------+---------+-----------------+



In [0]:
spark.sql(f"""
SELECT
    percentile(total_spent, array(0.25, 0.50, 0.75, 0.90, 0.95))
AS spending_distribution

FROM {GOLD_CUSTOMER_360}
""").show(truncate=False)

+---------------------------------------------------------------------------------+
|spending_distribution                                                            |
+---------------------------------------------------------------------------------+
|[3586.9775, 6677.0199999999995, 10396.682499999999, 14474.05, 17257.671499999993]|
+---------------------------------------------------------------------------------+



In [0]:
# Thresholds derived from Customer 360 spending percentile analysis
HIGH_VALUE_THRESHOLD = 10400
MEDIUM_VALUE_THRESHOLD = 6700
MIN_HIGH_VALUE_ORDERS = 5

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE} AS

SELECT

    customer_id,
    customer_city,
    customer_state,
    total_orders,
    total_items_purchased,
    total_spent,
    average_item_value,
    first_purchase,
    last_purchase,
    customer_lifetime_days,

    CASE
        WHEN total_spent >= {HIGH_VALUE_THRESHOLD }
             AND total_orders >= {MIN_HIGH_VALUE_ORDERS}
        THEN 'High Value'

        WHEN total_spent >= {MEDIUM_VALUE_THRESHOLD}
        THEN 'Medium Value'

        ELSE 'Low Value'

    END AS customer_segment

FROM {SOURCE_TABLE}
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
customer_segmentation_df = spark.table(TARGET_TABLE)
display(customer_segmentation_df.limit(10))

customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days,customer_segment
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809,High Value
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415,Medium Value
CUST_000066,Campo Grande,MS,4,4,4156.87,1039.2175,2021-06-09T23:00:00.000Z,2022-06-14T08:46:00.000Z,370,Low Value
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741,High Value
CUST_000095,Teresina,PI,1,1,1036.53,1036.53,2022-08-08T15:08:00.000Z,2022-08-08T15:08:00.000Z,0,Low Value
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559,Medium Value
CUST_000117,Manaus,AM,0,0,0.0,0.0,null,null,0,Low Value
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849,High Value
CUST_000136,Joao Pessoa,PB,1,3,3487.1,1162.3666666666666,2021-06-08T11:29:00.000Z,2021-06-08T11:29:00.000Z,0,Low Value
CUST_000159,Porto Velho,RO,2,2,2271.27,1135.635,2021-11-11T15:22:00.000Z,2022-06-21T01:37:00.000Z,222,Low Value


In [0]:
rows_written = customer_segmentation_df.count()

segment_summary = (
    customer_segmentation_df
    .groupBy("customer_segment")
    .count()
)

display(segment_summary)

print(f"Rows Written : {rows_written}")

customer_segment,count
High Value,2474
Medium Value,5003
Low Value,7523


Rows Written : 15000


In [0]:
null_segments = customer_segmentation_df.filter(
    "customer_segment IS NULL"
).count()

print(f"Null Segments : {null_segments}")

assert null_segments == 0

assert rows_written == customer_360_df.count()

Null Segments : 0


In [0]:
source = SOURCE_TABLE

bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=source,
    target=TARGET_TABLE,
    rows_read=customer_360_df.count(),
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS"
)

LOAD REPORT
Pipeline        : Gold Customer Segmentation
Run ID          : 7524cacf-0087-4c24-9d44-ee775afa7595
Source          : retailmart.gold.customer_360
Target          : retailmart.gold.customer_segmentation
Rows Read       : 15000
Rows Written    : 15000
Duplicate Rows  : 0
Start Time      : 2026-07-19 06:58:44.727001
End Time        : 2026-07-19 06:58:55.555831
Duration (sec)  : 10.83
Status          : SUCCESS


## Engineering Observations

• Customers were segmented using a CASE statement based on both spending behavior and purchase frequency, creating meaningful business categories.

• Business segmentation thresholds were derived after analyzing the customer spending distribution, ensuring that the classification reflects the characteristics of the underlying dataset rather than arbitrary values.

• Performing segmentation on the Gold Customer 360 table eliminated the need to repeatedly aggregate transactional data, improving query performance and maintaining a clean medallion architecture.

• The resulting segmentation table can be directly consumed by business intelligence dashboards, marketing campaigns, and customer retention analyses.

• This Gold table serves as a reusable analytical dataset for customer-focused reporting while keeping business logic centralized and easy to maintain.